In [ ]:
!pip install roboflow
! pip install ultralytics

  Using cached ultralytics-8.3.170-py3-none-any.whl.metadata (37 kB)
  Using cached ultralytics_thop-2.0.14-py3-none-any.whl.metadata (9.4 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-9.1.0.70-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.4.5.8-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.2.1.3-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.5.147-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.6.1.9-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 13.1 MB/s eta 0:00:00
   ━

# Get Dataset

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="ViGwERkCLbXkwyMOuthg")
project = rf.workspace("viren-dhanwani").project("tennis-ball-detection")
version = project.version(6)
dataset = version.download("yolov5")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to tennis-ball-detection-6 in yolov5pytorch:: 100%|██████████| 1168/1168 [00:00<00:00, 5066.17it/s]


In [ ]:
import shutil
shutil.move("tennis-ball-detection-6/train",
            "tennis-ball-detection-6/tennis_ball_detection-6/train"
            )

shutil.move("tennis-ball-detection-6/test",
            "tennis-ball-detection-6/tennis_ball_detection-6/test"
            )

shutil.move("tennis-ball-detection-6/valid",
            "tennis-ball-detection-6/tennis_ball_detection-6/valid"
            )

'tennis-ball-detection-6/tennis_ball_detection-6/valid'

In [ ]:
!yolo task=detect mode=train model=yolov5l6u.pt data={dataset.location}/data.yaml epochs=100 imgsz=640

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
100% 165M/165M [00:01<00:00, 165MB/s]
Ultralytics 8.3.170 🚀 Python-3.11.13 torch-2.6.0+cu124 CPU (Intel Xeon 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/tennis-ball-detection-6/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz

In [ ]:
!wget --header="Host: drive.usercontent.google.com" --header="User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3" --header="Accept: */*" --header="Accept-Language: en-US,en;q=0.8" --header="Accept-Encoding: gzip, deflate, sdch" --header="Connection: keep-alive" "https://drive.usercontent.google.com/download?id=1Zy4b7k2J6a8f3g4h5j6k7l8m9n0o1p2q&export=download&confirm=t&uuid=12345678-1234-1234-1234-123456789012" -O yolov5l6u.pt

--2025-07-29 17:23:44--  https://drive.usercontent.google.com/download?id=1Zy4b7k2J6a8f3g4h5j6k7l8m9n0o1p2q&export=download&confirm=t&uuid=12345678-1234-1234-1234-123456789012
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 142.251.107.132, 2607:f8b0:400c:c13::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|142.251.107.132|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2025-07-29 17:23:44 ERROR 404: Not Found.



In [ ]:
!unzip tennis_court_dataset.zip

unzip:  cannot find or open tennis_court_dataset.zip, tennis_court_dataset.zip.zip or tennis_court_dataset.zip.ZIP.


In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
import json
import cv2
import numpy as np

In [ ]:
devices = ["cuda:0" if torch.cuda.is_available() else "cpu"]

In [ ]:
class keypointDataset(Dataset):
    def __init__(self, img_dir, data_file):
        self.img_dir = img_dir
        with open(data_file, 'r') as f:
            self.data = json.load(f)

        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        img = cv2.imread(f"{self.img_dir}/{item['image']}")
        h, w = img.shape[:2]
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = self.transform(img)
        kps = np.array(item["kps"]).flatten()
        kps = kps.astype(np.float32)

        kps[::2] *= 224 / w # Scale x-coordinates
        kps[1::2] *= 224 / h # Scale y-coordinates

        return img, kps



In [ ]:
train_dataset = keypointDataset(
    "../tennis_court_det_dataset/data/images",
    "../tennis_court_det_dataset/data/data_train.json"
)

val_dataset = keypointDataset(
    "../tennis_court_det_dataset/data/images",
    "data_val.json"
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

FileNotFoundError: [Errno 2] No such file or directory: '../tennis_court_det_dataset/data/data_train.json'